# Đọc dataset ViMD cho bài toán ASR

Notebook này đọc dataset `nguyendv02/ViMD_Dataset`, kiểm tra các split và xem transcript/audio. Audio được giữ ở chế độ `decode=False` trong phần kiểm tra metadata; muốn lấy waveform cần cài thêm PyTorch + TorchCodec tương thích.

## Context & Methods

- Dataset: ViMD, gồm tiếng nói đa vùng miền tiếng Việt.
- Output kiểm tra chính: `text` và metadata; audio dùng `decode=False` để không tự giải mã waveform.
- Notebook không sửa dữ liệu gốc và không tải lại nếu Hugging Face đã có cache.

In [7]:
from pathlib import Path
from datasets import Audio, Dataset, DatasetDict, concatenate_datasets, load_dataset

DATASET_ID = "nguyendv02/ViMD_Dataset"
CACHE_ROOT = Path.home() / ".cache" / "huggingface" / "datasets"
cache_infos = sorted(
    (CACHE_ROOT / "nguyendv02___vi_md_dataset").rglob("dataset_info.json"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if cache_infos:
    # Đọc Arrow cache trực tiếp, không cần tải lại hoặc tạo cache lock mới.
    CACHE_DIR = cache_infos[0].parent
    dataset = DatasetDict({
        split: concatenate_datasets([
            Dataset.from_file(str(path))
            for path in sorted(CACHE_DIR.glob(f"vi_md_dataset-{split}-*.arrow"))
        ])
        for split in ("train", "valid", "test")
    })
    print(f"Loaded processed Arrow cache: {CACHE_DIR}")
else:
    # Máy chưa có cache sẽ tải từ Hugging Face và lưu vào cache mặc định.
    dataset = load_dataset(DATASET_ID, download_mode="reuse_dataset_if_exists")

dataset

Loaded processed Arrow cache: /home/fernir/.cache/huggingface/datasets/nguyendv02___vi_md_dataset/default/0.0.0/3a5b30157034e7eadd5c75fae1a820c6f9383398


DatasetDict({
    train: Dataset({
        features: ['region', 'province_code', 'province_name', 'filename', 'text', 'speakerID', 'gender', 'audio'],
        num_rows: 15023
    })
    valid: Dataset({
        features: ['region', 'province_code', 'province_name', 'filename', 'text', 'speakerID', 'gender', 'audio'],
        num_rows: 1900
    })
    test: Dataset({
        features: ['region', 'province_code', 'province_name', 'filename', 'text', 'speakerID', 'gender', 'audio'],
        num_rows: 2026
    })
})

## Data

In [2]:
split_summary = {
    split: {
        "rows": len(part),
        "columns": part.column_names,
        "features": str(part.features),
    }
    for split, part in dataset.items()
}
split_summary

{'train': {'rows': 15023,
  'columns': ['region',
   'province_code',
   'province_name',
   'filename',
   'text',
   'speakerID',
   'gender',
   'audio'],
  'features': "{'region': Value('string'), 'province_code': Value('int64'), 'province_name': Value('string'), 'filename': Value('string'), 'text': Value('string'), 'speakerID': Value('string'), 'gender': Value('int64'), 'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None)}"},
 'valid': {'rows': 1900,
  'columns': ['region',
   'province_code',
   'province_name',
   'filename',
   'text',
   'speakerID',
   'gender',
   'audio'],
  'features': "{'region': Value('string'), 'province_code': Value('int64'), 'province_name': Value('string'), 'filename': Value('string'), 'text': Value('string'), 'speakerID': Value('string'), 'gender': Value('int64'), 'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None)}"},
 'test': {'rows': 2026,
  'columns': ['region',
   'province_code

In [3]:
# Không decode audio: chỉ xem bytes/path và metadata văn bản.
dataset_no_decode = {
    split: part.cast_column("audio", Audio(decode=False))
    for split, part in dataset.items()
}
metadata_columns = [
    "region", "province_code", "province_name",
    "filename", "text", "speakerID", "gender"
]
preview = dataset_no_decode["train"].select_columns(metadata_columns).to_pandas().head(10)
preview

,region,province_code,province_name,filename,text,speakerID,gender
0,North,11,CaoBang,11_0001.wav,"Nghiên cứu học tập,các ứng dụng các khoa học c...",spk_11_0001,1
1,North,11,CaoBang,11_0002.wav,Chuyển đổi số hiện nay đang được là đang được ...,spk_11_0001,1
2,North,11,CaoBang,11_0003.wav,Trong quá trình mình làm việc để quản lý thông...,spk_11_0002,1
3,North,11,CaoBang,11_0004.wav,Ý tưởng đầu tiên tôi phải nhắc đến là vấn đề v...,spk_11_0002,1
4,North,11,CaoBang,11_0005.wav,giúp giải quyết đa dạng các nội dung trong quả...,spk_11_0002,1
5,North,11,CaoBang,11_0006.wav,Em rất là tự hào vì năm đầu tiên em được tham ...,spk_11_0003,0
6,North,11,CaoBang,11_0007.wav,"Em cảm thấy rất là tự hào và qua cái lễ hội, l...",spk_11_0004,0
7,North,11,CaoBang,11_0008.wav,Năm nay thì Cao Trương tôi thì cũng chuẩn bị n...,spk_11_0005,1
8,North,11,CaoBang,11_0009.wav,Em thưa chị là em cảm thấy rất là vinh dự và t...,spk_11_0006,0
9,North,11,CaoBang,11_0010.wav,"Qua thực tế, thì cán bộ công nhân viên từ kỹ t...",spk_11_0007,1


In [4]:
sample = dataset_no_decode["train"][0]
audio_info = sample["audio"]
{
    "filename": sample["filename"],
    "text": sample["text"],
    "audio_path": audio_info.get("path"),
    "audio_bytes": len(audio_info.get("bytes", b"")),
}

{'filename': '11_0001.wav',
 'text': 'Nghiên cứu học tập,các ứng dụng các khoa học công nghệ và những dụng tiên tiến để đưa vào trong áp dụng trong công việc của mình. Từ đó giúp chuyển đổi những cái khó khăn trong công việc, đưa ra những thuận lợi và những cái nhanh gọn hơn giải quyết các thủ tục hành chính cho được thuận tiện hơn.',
 'audio_path': '11_0001.wav',
 'audio_bytes': 4276684}

In [6]:
from IPython.display import Audio as NotebookAudio, display

# Phát mẫu đầu tiên trong notebook; không cần decode bằng torchcodec.
display(NotebookAudio(data=audio_info["bytes"], autoplay=False))

## Results

Các kiểm tra chi tiết hơn về null, speaker leakage, định dạng WAV và tổng thời lượng đã được thực hiện trong [audit report](../docs/vimd_dataset_audit.md).

In [5]:
assert set(dataset.keys()) == {"train", "valid", "test"}
assert all(len(part) > 0 for part in dataset.values())
assert all("text" in part.column_names and "audio" in part.column_names for part in dataset.values())
print("Dataset loaded successfully")

Dataset loaded successfully


## Takeaways

Sau khi chạy notebook, có thể dùng `dataset_no_decode` để kiểm tra transcript/metadata. Khi huấn luyện ASR, cần bổ sung backend giải mã audio và chuẩn hóa waveform về mono 16 kHz trong pipeline preprocessing.